In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt
import uuid

# Define the state with two variables
class State(TypedDict):
    number: int
    word: str
    feedback : str

# Node 1: double the number
def double_number(state: State) -> State:
    state["number"] = state["number"] * 2
    return state

def interrupt_state(state:State):
    interrupt("The state has been interrupted")

def resume_state(state:State):
    

# Node 2: uppercase the word
def uppercase_word(state: State) -> State:
    state["word"] = state["word"].upper()
    return state

# Build the graph
builder = StateGraph(State)
builder.add_node("double_number", double_number)
builder.add_node("uppercase_word", uppercase_word)

# Connect edges
builder.add_edge(START, "double_number")
builder.add_edge("double_number", "uppercase_word")
builder.add_edge("uppercase_word", END)

# Create a memory checkpointer
memory = MemorySaver()

# Compile the graph with checkpointer
graph = builder.compile(checkpointer=memory)

# Create a config with a thread_id
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Run the graph with config
result = graph.invoke({"number": 5, "word": "hello"}, config=config)
print("Thread:", config["configurable"]["thread_id"])
print("Result:", result)   # {'number': 10, 'word': 'HELLO'}

# 🔹 Example: get state after execution
state = graph.get_state(config)
print("Saved state:", state.values)


In [ ]:
# Update the state with new feedback
graph.update_state(config, {"feedback": "abc"})

# Retrieve updated state
updated_state = graph.get_state(config)
print("Updated state:", updated_state.values)


In [ ]:
state.values

In [ ]:
from openai import OpenAI
import os
 # should NOT be None
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "ping"}]
)
print(resp.choices[0].message.content)
